In [38]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"

fig = go.Figure()

# Axis lines
for axis, color, label in [
    ([2,0,0], 'red',   'X'),
    ([0,2,0], 'green', 'Y'),
    ([0,0,2], 'blue',  'Z')
]:
    fig.add_trace(go.Scatter3d(
        x=[0, axis[0]], y=[0, axis[1]], z=[0, axis[2]],
        mode='lines+text',
        line=dict(color=color, width=4),
        text=['', label],
        textposition='top center',
        showlegend=False
    ))

fig.update_layout(
    scene=dict(
        xaxis_title='X', yaxis_title='Y', zaxis_title='Z',
        bgcolor='rgb(10,10,20)',
        xaxis=dict(gridcolor='rgba(255,255,255,0.1)', color='white'),
        yaxis=dict(gridcolor='rgba(255,255,255,0.1)', color='white'),
        zaxis=dict(gridcolor='rgba(255,255,255,0.1)', color='white'),
    ),
    paper_bgcolor='rgb(10,10,20)',
    font=dict(color='white'),
    title="3D Vector Space",
    margin=dict(l=0, r=0, t=40, b=0)
)


def plot_vector(vec, label: str = "v", color: str = "yellow",base=np.array([0,0,0])):
    """
    Plot a vector from the origin.

    Args:
        vec   : [x, y, z] coordinates
        label : display name
        color : any CSS color string
    """
    v = vec.flatten()
    base=base.flatten()
    mag = np.linalg.norm(v)

    fig.add_trace(go.Scatter3d(
        x=[base[0], v[0]], y=[base[1], v[1]], z=[base[2], v[2]],
        mode='lines+markers+text',
        line=dict(color=color, width=5),
        marker=dict(size=[0, 8], color=color, symbol=['circle', 'diamond']),
        text=['', label],
        textposition='top center',
        textfont=dict(color=color, size=13),
        name=label,
        hovertemplate=f"<b>{label}</b><br>({v[0]:.2f}, {v[1]:.2f}, {v[2]:.2f})<br>Magnitude: {mag:.3f}<extra></extra>"
    ))

    print(f"[{label}] ({v[0]:.3f}, {v[1]:.3f}, {v[2]:.3f}) | magnitude: {mag:.3f}")


def plot_span(matrix, label: str = "span", color: str = "cyan", opacity: float = 0.4, grid_res: int = 15):
    """
    Visualizes the subspace spanned by the columns of an MxN matrix in R^3.
    Automatically extracts up to 3 independent basis vectors via SVD.

    Args:
        matrix  : numpy array of shape (m, n) where m >= 3
        label   : display name
        color   : surface/line color
        opacity : transparency (0-1)
        grid_res: resolution of the surface grid
    """
    M = np.array(matrix, dtype=float)
    if M.ndim == 1:
        M = M.reshape(-1, 1)

    assert M.shape[0] >= 3, "Matrix must have at least 3 rows (we live in R^3)"

    rank = np.linalg.matrix_rank(M)
    effective_rank = min(rank, 3)

    # SVD gives the most stable orthonormal basis for the column space
    # U columns = orthonormal basis vectors spanning the column space in R^m
    # we only need the first 3 rows (x, y, z) of each basis vector
    U, _, _ = np.linalg.svd(M, full_matrices=False)
    basis = [U[:3, i] for i in range(effective_rank)]

    print(f"[{label}] shape={M.shape} | rank={rank} | effective_rank={effective_rank} → ", end="")

    # --- RANK 1 : line ---
    if effective_rank == 1:
        print("spanning a line")
        d = basis[0]
        t = np.linspace(-2, 2, 100)
        pts = np.outer(t, d)
        fig.add_trace(go.Scatter3d(
            x=pts[:,0], y=pts[:,1], z=pts[:,2],
            mode='lines',
            line=dict(color=color, width=6),
            name=label,
            hovertemplate=f"<b>{label}</b> (line)<extra></extra>"
        ))

    # --- RANK 2 : plane ---
    elif effective_rank == 2:
        print("spanning a plane")
        u, v = basis[0], basis[1]
        s = np.linspace(-2, 2, grid_res)
        t = np.linspace(-2, 2, grid_res)
        S, T = np.meshgrid(s, t)
        X = S * u[0] + T * v[0]
        Y = S * u[1] + T * v[1]
        Z = S * u[2] + T * v[2]
        fig.add_trace(go.Surface(
            x=X, y=Y, z=Z,
            opacity=opacity,
            colorscale=[[0, color], [1, color]],
            showscale=False,
            name=label,
            hovertemplate=f"<b>{label}</b> (plane)<extra></extra>"
        ))

    # --- RANK 3 : all of R^3 ---
    elif effective_rank == 3:
        print("spanning all of R^3")
        r = 2
        verts = np.array([[x,y,z]
                          for x in [-r,r] for y in [-r,r] for z in [-r,r]], dtype=float)
        faces = [
            [0,1,3],[0,3,2],
            [4,5,7],[4,7,6],
            [0,1,5],[0,5,4],
            [2,3,7],[2,7,6],
            [0,2,6],[0,6,4],
            [1,3,7],[1,7,5],
        ]
        fig.add_trace(go.Mesh3d(
            x=verts[:,0], y=verts[:,1], z=verts[:,2],
            i=[f[0] for f in faces],
            j=[f[1] for f in faces],
            k=[f[2] for f in faces],
            opacity=opacity * 0.5,
            color=color,
            name=label,
            hovertemplate=f"<b>{label}</b> (all R³)<extra></extra>"
        ))



In [39]:
# base vector is either a line or a basis
base_vector = np.array([
    [1],
    [0],
    [0]
    ],dtype=float)
#vector b can be anything in R^3
vector_b = np.array([1,1,1],dtype=float)
vector_b = vector_b.reshape(-1,1)

plot_span(base_vector,"base_vector", "pink")
plot_vector(vector_b,"vector_b", "orange")

[base_vector] shape=(3, 1) | rank=1 | effective_rank=1 → spanning a line
[vector_b] (1.000, 1.000, 1.000) | magnitude: 1.732


In [40]:
def inverse_matrix(mat):
    """
    Returns the inverse of a given square matrix if it exists.
    Raises ValueError if the matrix is not square or is singular.
    """
    flat = mat.flatten()
    if len(flat)==1:
        return flat[0]

    # Check if matrix is square
    if mat.shape[0] != mat.shape[1]:
        raise ValueError("Matrix must be square to find its inverse.")

    # Check if matrix is singular (determinant = 0)
    det = np.linalg.det(mat)
    if np.isclose(det, 0):
        raise ValueError("Matrix is singular and cannot be inverted.")

    # Compute inverse
    return np.linalg.inv(mat)



In [41]:
if len(base_vector[0])>1:
    projection_vector = (base_vector@inverse_matrix(base_vector.T @ base_vector))@base_vector.T@vector_b
else:
    projection_vector = (base_vector@base_vector.T)*inverse_matrix(base_vector.T @ base_vector)@vector_b

plot_vector(projection_vector,"projection_vector", "cyan")
error_vector = vector_b-projection_vector
plot_vector(error_vector,"error_vector", "white")

[projection_vector] (1.000, 0.000, 0.000) | magnitude: 1.000
[error_vector] (0.000, 1.000, 1.000) | magnitude: 1.414


In [42]:
# zero implies it is ERROR_VECTOR is perpendicular to base and projection vector
error_vector.T@base_vector

array([[0.]])

In [43]:
fig.show()